In [ ]:
import numpy as np
import pickle
import pandas as pd
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr, pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from predict_associations_comb_medication import predict_dda, calculate_metrics, Split_dataset 

In [ ]:
DF = pd.read_csv('/home/yin/DREAMwalk-main/DREAMwalk-main/demo/drug_combination/combination_samples.csv')
normalized_columns = (DF.iloc[:, 3:] - DF.iloc[:, 3:].min()) / (DF.iloc[:, 3:].max() - DF.iloc[:, 3:].min())
DF.iloc[:,3:] = normalized_columns
DF.to_csv('/home/yin/DREAMwalk-main/DREAMwalk-main/demo/drug_combination/normalized_data.csv',index = None)

In [ ]:
results = pd.DataFrame(columns=['pathsave', 'mse', 'rmse', 'mae', 'r2', 'Pearson', 'Spearman'])
path = []
for t in ['T1', 'T2','T3']:
    for n in ['N1', 'N2','N3']:
        for w in ['W1', 'W2','W3']:
            path_save = t+n+w
            path.append(path_save)
for pathsave in path:
    embeddingf='/home/yin/DREAMwalk-main/DREAMwalk-main/data/data_jiaqi_stitch_cutoff_merge/{}/embedding_file_yin.pkl'.format(pathsave)
    disease_herb_label_f = '/home/yin/DREAMwalk-main/DREAMwalk-main/demo/LiuRui/data/Combination.csv' 
    X,y = Split_dataset(pairf=disease_herb_label_f,embeddingf=embeddingf)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    regressor = RandomForestRegressor(n_estimators=100,  
                                  max_depth=None,    
                                  min_samples_split=2,  
                                  random_state=42)
    regressor.fit(X_train, y_train)
    y_pred = regressor.predict(X_test)
    mse, rmse, mae, r2, pearson_corr, spearman_corr = calculate_metrics(y_test, y_pred)
    results = results.append({'pathsave': pathsave, 'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2, 'Pearson': pearson_corr, 'Spearman': spearman_corr}, ignore_index=True)
results.to_csv('/home/yin/DREAMwalk-main/DREAMwalk-main/demo/LiuRui/Evaluate_Comb_parameters/RF_Evaluate_parameters.csv', index=False)
